In [1]:
# from numba import cuda
import paicos as pa
import numpy as np
import cupy as cp
import turbocluster as tc
import math


# A snapshot object
# snap = pa.Snapshot(pa.data_dir, 247)
snap = pa.Snapshot('/lustre/astro/berlok/zoom-simulations-new-ics/halo_0003/adiabatic-mhd/zoom4_ics_v1/output', 247)
# snap = pa.Snapshot('/lustre/astro/berlok/zoom-simulations-new-ics/halo_0003/tng/zoom12_ics_v1/output', 247)
center = snap.Cat.Group['GroupPos'][0]
widths = np.array([1000., 500., 750.], dtype=float)

# m_filter = 1000*snap.mass
# filter_length = (np.cbrt(3*m_filter/(4*np.pi*snap['0_Density']))).arepo
filter_length = 2*snap['0_Diameters']


Attempting to get derived variable: 0_Diameters...
	So we need the variable: 0_Volume...	[DONE]



In [2]:
sf = tc.SmoothingFilter(snap, center, widths, npix=64, orientation=None, 
                        search_radius=filter_length.value)

In [3]:
filt_density = sf.filter_variable('0_Density',filter_length,weight='0_Volume', 
                                  filter_type="gaussian")

In [4]:
filt_density_shared = sf.filter_variable('0_Density',filter_length,weight='0_Volume', 
                                         filter_type="gaussian", shared_mem=True, Nmax=64)

numBlocksInDomain = 71819


In [5]:
filt_density_optimized = sf.filter_variable('0_Density',filter_length,weight='0_Volume', 
                                            filter_type="gaussian", optimized=True)

In [6]:
if not np.allclose(filt_density_shared.value, filt_density.value, rtol=1e-12, atol=1e-8):
    print('ERROR: the results of the shared memory filter and the base filter \
          are not identical (within tolerance)')

In [7]:
if not np.allclose(filt_density_optimized.value, filt_density.value, rtol=1e-12, atol=1e-8):
    print('ERROR: the results of the optimized filter and the base filter \
          are not identical (within tolerance)')